In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.kkbox
COMMENT 'KKBOX subscription churn and cohort analysis project';

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.kkbox.raw_data
COMMENT 'Raw KKBOX source files';

In [0]:
train_path = "/Volumes/workspace/kkbox/raw_data/train.csv"

train_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(train_path)
)

display(train_df.limit(10))

In [0]:
from pyspark.sql import functions as F

validation = train_df.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("msno").alias("unique_customers"),
    F.sum(F.when(F.col("msno").isNull(), 1).otherwise(0)).alias("null_customer_ids"),
    F.sum(F.when(F.col("is_churn").isNull(), 1).otherwise(0)).alias("null_churn_labels")
)

display(validation)

In [0]:
churn_distribution = (
    train_df
    .groupBy("is_churn")
    .count()
    .withColumnRenamed("count", "customer_count")
    .orderBy("is_churn")
)

display(churn_distribution)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Target customer counts based on the original churn distribution
non_churn_target = 187215
churn_target = 12785

# Create a deterministic hash for reproducible selection
scoped_df = (
    train_df
    .withColumn("selection_hash", F.sha2(F.col("msno"), 256))
)

# Rank customers separately within each churn class
window_spec = Window.partitionBy("is_churn").orderBy("selection_hash")

scoped_df = (
    scoped_df
    .withColumn("selection_rank", F.row_number().over(window_spec))
)

# Select exactly 200,000 customers
customer_scope = (
    scoped_df
    .filter(
        ((F.col("is_churn") == 0) & (F.col("selection_rank") <= non_churn_target))
        |
        ((F.col("is_churn") == 1) & (F.col("selection_rank") <= churn_target))
    )
    .select("msno", "is_churn")
)

display(customer_scope.groupBy("is_churn").count().orderBy("is_churn"))

In [0]:
customer_scope.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.kkbox.customer_scope")

In [0]:
saved_scope = spark.table("workspace.kkbox.customer_scope")

display(
    saved_scope.groupBy("is_churn")
    .count()
    .orderBy("is_churn")
)

In [0]:
members_path = "/Volumes/workspace/kkbox/raw_data/members_v3.csv"

members_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(members_path)
)

display(members_df.limit(10))

In [0]:
members_validation = members_df.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("msno").alias("unique_customers"),
    F.sum(F.when(F.col("msno").isNull(), 1).otherwise(0)).alias("null_customer_ids"),
    F.sum(F.when(F.col("city").isNull(), 1).otherwise(0)).alias("null_city"),
    F.sum(F.when(F.col("bd").isNull(), 1).otherwise(0)).alias("null_age"),
    F.sum(F.when(F.col("gender").isNull(), 1).otherwise(0)).alias("null_gender"),
    F.sum(F.when(F.col("registered_via").isNull(), 1).otherwise(0)).alias("null_registered_via"),
    F.sum(F.when(F.col("registration_init_time").isNull(), 1).otherwise(0)).alias("null_registration_date")
)

display(members_validation)

In [0]:
from pyspark.sql import functions as F

customer_scope = spark.table("workspace.kkbox.customer_scope")

members_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load("/Volumes/workspace/kkbox/raw_data/members_v3.csv")
)

scope_members = (
    customer_scope
    .join(
        members_df,
        on="msno",
        how="left"
    )
)

display(
    scope_members.agg(
        F.count("*").alias("scope_customers"),
        F.sum(
            F.when(F.col("city").isNotNull(), 1).otherwise(0)
        ).alias("customers_with_member_data"),
        F.sum(
            F.when(F.col("city").isNull(), 1).otherwise(0)
        ).alias("customers_missing_member_data")
    )
)

In [0]:
transactions_path = "/Volumes/workspace/kkbox/raw_data/transactions_v2.csv"

transactions_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(transactions_path)
)

display(transactions_df.limit(10))

In [0]:
transactions_validation = transactions_df.agg(
    F.count("*").alias("total_rows"),
    F.countDistinct("msno").alias("unique_customers"),
    F.sum(
        F.when(F.col("msno").isNull(), 1).otherwise(0)
    ).alias("null_customer_ids"),
    F.sum(
        F.when(F.col("payment_plan_days").isNull(), 1).otherwise(0)
    ).alias("null_plan_days"),
    F.sum(
        F.when(F.col("plan_list_price").isNull(), 1).otherwise(0)
    ).alias("null_plan_price"),
    F.sum(
        F.when(F.col("actual_amount_paid").isNull(), 1).otherwise(0)
    ).alias("null_amount_paid"),
    F.sum(
        F.when(F.col("transaction_date").isNull(), 1).otherwise(0)
    ).alias("null_transaction_date"),
    F.sum(
        F.when(F.col("membership_expire_date").isNull(), 1).otherwise(0)
    ).alias("null_expiry_date")
)

display(transactions_validation)

In [0]:
customer_scope = spark.table("workspace.kkbox.customer_scope")

transaction_scope = (
    customer_scope
    .select("msno")
    .join(
        transactions_df.select("msno").distinct(),
        on="msno",
        how="left"
    )
)

display(
    transaction_scope.agg(
        F.count("*").alias("scope_customers"),
        F.sum(
            F.when(F.col("msno").isNotNull(), 1).otherwise(0)
        ).alias("customers_with_transactions")
    )
)

In [0]:
from pyspark.sql import functions as F

transaction_anomalies = transactions_df.agg(
    # Payment and pricing checks
    F.sum(
        F.when(F.col("actual_amount_paid") <= 0, 1).otherwise(0)
    ).alias("non_positive_amount_paid"),

    F.sum(
        F.when(F.col("plan_list_price") < 0, 1).otherwise(0)
    ).alias("negative_plan_price"),

    F.sum(
        F.when(F.col("payment_plan_days") <= 0, 1).otherwise(0)
    ).alias("non_positive_plan_days"),

    # Flag value checks
    F.sum(
        F.when(~F.col("is_auto_renew").isin(0, 1), 1).otherwise(0)
    ).alias("invalid_auto_renew"),

    F.sum(
        F.when(~F.col("is_cancel").isin(0, 1), 1).otherwise(0)
    ).alias("invalid_cancel"),

    # Date relationship
    F.sum(
        F.when(
            F.to_date("transaction_date", "yyyyMMdd") >
            F.to_date("membership_expire_date", "yyyyMMdd"),
            1
        ).otherwise(0)
    ).alias("transaction_after_expiry")
)

display(transaction_anomalies)

In [0]:
non_positive_payments = (
    transactions_df
    .filter(F.col("actual_amount_paid") <= 0)
    .select(
        "msno",
        "payment_method_id",
        "payment_plan_days",
        "plan_list_price",
        "actual_amount_paid",
        "is_auto_renew",
        "transaction_date",
        "membership_expire_date",
        "is_cancel"
    )
)

display(non_positive_payments.limit(20))

In [0]:
zero_payment_summary = (
    transactions_df
    .filter(F.col("actual_amount_paid") == 0)
    .groupBy(
        "plan_list_price",
        "payment_plan_days",
        "is_auto_renew",
        "is_cancel"
    )
    .count()
    .orderBy(F.desc("count"))
)

display(zero_payment_summary.limit(30))

In [0]:
invalid_plan_days = (
    transactions_df
    .filter(F.col("payment_plan_days") <= 0)
    .select(
        "msno",
        "payment_method_id",
        "payment_plan_days",
        "plan_list_price",
        "actual_amount_paid",
        "is_auto_renew",
        "transaction_date",
        "membership_expire_date",
        "is_cancel"
    )
)

display(invalid_plan_days.limit(20))

In [0]:
invalid_plan_summary = (
    transactions_df
    .filter(F.col("payment_plan_days") <= 0)
    .groupBy(
        "payment_method_id",
        "plan_list_price",
        "actual_amount_paid",
        "is_auto_renew",
        "is_cancel"
    )
    .count()
    .orderBy(F.desc("count"))
)

display(invalid_plan_summary.limit(30))

In [0]:
invalid_plan_summary_check = (
    invalid_plan_summary
    .agg(
        F.sum("count").alias("total_grouped_rows"),
        F.count("*").alias("number_of_combinations")
    )
)

display(invalid_plan_summary_check)

In [0]:
expiry_anomaly_summary = (
    transactions_df
    .filter(
        F.to_date("transaction_date", "yyyyMMdd") >
        F.to_date("membership_expire_date", "yyyyMMdd")
    )
    .groupBy(
        "payment_method_id",
        "payment_plan_days",
        "plan_list_price",
        "actual_amount_paid",
        "is_auto_renew",
        "is_cancel"
    )
    .count()
    .orderBy(F.desc("count"))
)

display(expiry_anomaly_summary.limit(30))

In [0]:
transaction_date_range = (
    transactions_df
    .agg(
        F.min(F.to_date("transaction_date", "yyyyMMdd")).alias("earliest_transaction_date"),
        F.max(F.to_date("transaction_date", "yyyyMMdd")).alias("latest_transaction_date"),
        F.min(F.to_date("membership_expire_date", "yyyyMMdd")).alias("earliest_expiry_date"),
        F.max(F.to_date("membership_expire_date", "yyyyMMdd")).alias("latest_expiry_date")
    )
)

display(transaction_date_range)

In [0]:
future_expiry_summary = (
    transactions_df
    .filter(
        F.to_date("membership_expire_date", "yyyyMMdd") > F.lit("2020-01-01")
    )
    .groupBy(
        "payment_method_id",
        "payment_plan_days",
        "plan_list_price",
        "actual_amount_paid",
        "is_auto_renew",
        "is_cancel"
    )
    .count()
    .orderBy(F.desc("count"))
)

display(future_expiry_summary.limit(30))

In [0]:
future_expiry_dates = (
    transactions_df
    .filter(
        F.to_date("membership_expire_date", "yyyyMMdd") > F.lit("2020-01-01")
    )
    .groupBy("membership_expire_date")
    .count()
    .orderBy(F.desc("count"))
)

display(future_expiry_dates.limit(20))

In [0]:
future_max_date = (
    transactions_df
    .filter(
        F.to_date("membership_expire_date", "yyyyMMdd") ==
        F.lit("2036-10-15")
    )
    .select(
        "msno",
        "payment_method_id",
        "payment_plan_days",
        "plan_list_price",
        "actual_amount_paid",
        "is_auto_renew",
        "is_cancel",
        "transaction_date",
        "membership_expire_date"
    )
)

display(future_max_date)

In [0]:
id_check = (
    customer_scope
    .filter(
        F.col("msno") == "F9//z42OpKq9GUSUNuSWmKIIww4deT2elqiqfSxznK0="
    )
)

display(id_check)

In [0]:
transaction_duplicate_check = (
    transactions_df
    .groupBy(
        "msno",
        "payment_method_id",
        "payment_plan_days",
        "plan_list_price",
        "actual_amount_paid",
        "is_auto_renew",
        "transaction_date",
        "membership_expire_date",
        "is_cancel"
    )
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("duplicate_groups"),
        F.sum("count").alias("rows_in_duplicate_groups")
    )
)

display(transaction_duplicate_check)

In [0]:
from pyspark.sql import functions as F

customer_scope = spark.table("workspace.kkbox.customer_scope")

transactions_clean = (
    transactions_df
    .join(
        customer_scope.select("msno"),
        on="msno",
        how="inner"
    )
    .select(
        "msno",
        "payment_method_id",
        "payment_plan_days",
        "plan_list_price",
        "actual_amount_paid",
        "is_auto_renew",
        "transaction_date",
        "membership_expire_date",
        "is_cancel",
        F.to_date(
            "transaction_date",
            "yyyyMMdd"
        ).alias("transaction_date_clean"),
        F.to_date(
            "membership_expire_date",
            "yyyyMMdd"
        ).alias("membership_expire_date_clean")
    )
)

display(transactions_clean.limit(10))

In [0]:
transactions_clean_validation = transactions_clean.agg(
    F.count("*").alias("total_transaction_rows"),
    F.countDistinct("msno").alias("unique_customers"),
    F.sum(
        F.when(F.col("transaction_date_clean").isNull(), 1).otherwise(0)
    ).alias("null_transaction_dates"),
    F.sum(
        F.when(F.col("membership_expire_date_clean").isNull(), 1).otherwise(0)
    ).alias("null_expiry_dates")
)

display(transactions_clean_validation)

In [0]:
customers_without_transactions = (
    customer_scope
    .select("msno", "is_churn")
    .join(
        transactions_df.select("msno").distinct(),
        on="msno",
        how="left_anti"
    )
)

display(
    customers_without_transactions.agg(
        F.count("*").alias("customers_without_transactions"),
        F.sum(
            F.when(F.col("is_churn") == 1, 1).otherwise(0)
        ).alias("churn_customers_without_transactions"),
        F.sum(
            F.when(F.col("is_churn") == 0, 1).otherwise(0)
        ).alias("non_churn_customers_without_transactions")
    )
)

In [0]:
transaction_scope_check = (
    customer_scope
    .join(
        transactions_df.select("msno").distinct(),
        on="msno",
        how="left"
    )
    .withColumn(
        "has_transaction_history",
        F.when(F.col("msno").isNotNull(), 1).otherwise(0)
    )
)

display(
    transaction_scope_check.groupBy("has_transaction_history")
    .count()
    .orderBy("has_transaction_history")
)

In [0]:
transactions_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.kkbox.transactions_clean")

In [0]:
saved_transactions = spark.table("workspace.kkbox.transactions_clean")

display(
    saved_transactions.agg(
        F.count("*").alias("total_transaction_rows"),
        F.countDistinct("msno").alias("unique_customers")
    )
)